# Indian Laws RAG Pipeline

End-to-end RAG system over the [Indian-Laws](https://huggingface.co/datasets/mratanusarkar/Indian-Laws) dataset.

**Stack**
- Orchestration: LangGraph
- Embeddings: mxbai-embed-large-v1 (dense) + bm25 (sparse)
- Vector DB: Qdrant
- Retrieval optimizations: RRF fusion, MMR, adaptive k, metadata pre-filtering, query decomposition, small-to-big context
- Reranker: BGE-reranker-v2-m3
- Web search fallback (low confidence): Serper API, scoped to indiacode.nic.in / indiankanoon.org
- Memory: SQLite (conversation history) + query rewriter
- Generator: OpenAI / Groq / Gemini switcher
- Eval: DeepEval

**Secrets required** (set via Colab Secrets or environment variables):
`OPENAI_API_KEY`, `GROQ_API_KEY`, `GEMINI_API_KEY`, `SERPER_API_KEY`

**How to read this notebook:** every markdown section explains *what* the upcoming method does, *why*
it's the right tool here, and (where useful) a tiny worked example. Every code cell is commented
line-by-line for syntax/mechanics, not just repeated in English — read the markdown for "why", the
code comments for "how".


## 1. Setup

In [2]:
# Install every third-party package this notebook touches.
# -q = quiet install (suppress the pip progress spam in Colab output)
# The trailing "\" lets the command continue on the next line -- it's still one shell command.
!pip install -q langgraph langchain langchain-community qdrant-client sentence-transformers fastembed \
    pymongo openai groq google-generativeai datasets deepeval requests

^C
ERROR: Operation cancelled by user


In [ ]:
import os # read environment variables (used as a secrets fallback outside Colab)
import re # regex -- imported for use elsewhere in experimentation; not required by the
# code paths below, kept because chunking/cleanup logic often needs it
import json # convert Python objects <-> JSON strings (used for LLM structured output, SQLite storage)
import hashlib # deterministic hashing -- used below to build stable document IDs
import numpy as np # vector math (cosine similarity, argmax) for MMR re-ranking

# TypedDict: lets us declare a plain dict's expected keys/types (used for LangGraph's state schema).
# Literal: restricts a variable to a fixed set of string values (used for the conditional-edge router).
# Optional[X] is shorthand for "X or None".
from typing import TypedDict, Literal, Optional

from datasets import load_dataset # HuggingFace `datasets` loader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # chunking utility, see Section 2

In [ ]:
# Secrets: works both in Colab (userdata) and elsewhere (os.environ) -- use whichever is available.
try:
    # google.colab is only importable when this notebook is actually running inside Colab.
    from google.colab import userdata
    def get_secret(name):
        return userdata.get(name) # pulls from Colab's "Secrets" panel (the key icon in the sidebar)
except ImportError:
    # Anywhere else (local Jupyter, a script, CI) fall back to plain OS environment variables.
    def get_secret(name):
        return os.environ.get(name)

OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
# GROQ_API_KEY = get_secret("GROQ_API_KEY")      # uncomment if you want the Groq branch of the generator
# GEMINI_API_KEY = get_secret("GEMINI_API_KEY")  # uncomment if you want the Gemini branch of the generator
SERPER_API_KEY = get_secret("SERPER_API_KEY")

## 2. Load and chunk the dataset

**The method:** each dataset row is one law section (`act_title`, `section`, `law` text). We treat
`(act_title, section)` as the real primary key, and generate a short deterministic id (`make_doc_id`)
by hashing that pair -- same input always produces the same id, so re-running this cell twice upserts
the *same* points instead of creating duplicates in Qdrant.

Some rows are unusable (empty `law` text, or missing `act_title`) and get dropped. Other rows are
*oversized* -- usually because the scraper captured a whole chapter instead of a single section --
and need to be split, otherwise a single embedding vector would have to represent way too much text
to be useful for similarity search.

**Small-to-big chunking:** instead of splitting and losing context, oversized rows are split into
smaller "child" chunks for embedding/retrieval, but every child still carries a `parent_doc_id`
pointing back to the *original, un-split* section text (`parent_text`). At answer time we retrieve
using the small, precise child chunk, but hand the LLM the full parent section -- small-to-big.

*Example:* Section 375 of the IPC (rape, definitions) is long enough to get split into 3 children by
`RecursiveCharacterTextSplitter`. All 3 children share `parent_doc_id = sha1("IPC::375")[:16]`. If the
child about clause (a) is the best semantic match for a query, we still return the *entire* Section 375
text to the LLM, not just clause (a) in isolation -- so the model has full context to answer correctly.

The splitter's `separators` list is ordered from "nicest cut" to "worst cut": it always tries the
first separator that actually appears in the text, and only falls back to a smaller unit (down to a
single space or character) if nothing bigger fits within `chunk_size`. That's why paragraph breaks and
numbered sub-clauses like `\n(1)` are listed before plain spaces -- legal text is naturally structured
around these markers, so splitting on them keeps clauses intact.


In [ ]:
MAX_CHUNK_CHARS = 2000   # max characters per chunk fed to the splitter (embedding models have a token limit)
CHUNK_OVERLAP = 200      # characters repeated between consecutive chunks, so a sentence split across
                         # a chunk boundary still appears in full in at least one chunk
MIN_LAW_CHARS = 5        # rows with less "law" text than this are almost certainly scraping junk -> dropped

# Ordered "prefer this cut point" list for RecursiveCharacterTextSplitter.
# The splitter tries separators top-to-bottom and only moves to the next one if a piece is still
# too big after splitting on the current one. Blank string "" at the end means "just hard-cut the text".
LEGAL_SEPARATORS = [
    "\n\n",                                            # paragraph breaks first (biggest, cleanest cut)
    "\n(1)", "\n(2)", "\n(3)", "\n(4)", "\n(5)",        # numbered sub-clauses common in Indian statutes
    "\n(a)", "\n(b)", "\n(c)", "\n(d)", "\n(e)",        # lettered sub-clauses
    "\n", ". ", " ", "",                                # line breaks, then sentences, then words, then chars
]

def make_doc_id(act_title: str, section: str) -> str:
    # sha1(...) -> a 40-character hex digest; deterministic for the same input string.
    # We only keep the first 16 hex chars -- plenty of entropy for a dataset this size, and shorter ids
    # are easier to read in logs/payloads. f"{act_title}::{section}" is an f-string: variables are
    # interpolated directly into the string at "{}" positions.
    return hashlib.sha1(f"{act_title}::{section}".encode("utf-8")).hexdigest()[:16]

def strip_boilerplate(act_title: str, text: str) -> str:
    # Some rows repeat the act title as the first line of the law text itself (a scraping artifact).
    # str.startswith(...) checks a prefix match; if present, slice it off and strip leading whitespace.
    if text.startswith(act_title):
        text = text[len(act_title):].lstrip()
    return text

def load_and_chunk(split: str = "train", num_samples: int = 500):
    # load_dataset(...) downloads/opens the HuggingFace dataset and returns a `Dataset` object,
    # which behaves like a list of dict-like rows.
    ds = load_dataset("mratanusarkar/Indian-Laws", split=split)
    # .select(range(...)) takes only the first `num_samples` rows -- keeps the demo fast to run.
    # min(...) guards against asking for more rows than the dataset actually has.
    ds = ds.select(range(min(num_samples, len(ds))))
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=MAX_CHUNK_CHARS, chunk_overlap=CHUNK_OVERLAP, separators=LEGAL_SEPARATORS,
    )

    records = []
    dropped = 0
    for row in ds:                                        # iterating a HF Dataset yields plain Python dicts
        # row.get("act_title") returns None if the key is missing/null; `or ""` swaps None -> "" so
        # .strip() never crashes on a None value.
        act_title = (row.get("act_title") or "").strip()
        section = (row.get("section") or "").strip()
        law = (row.get("law") or "").strip()

        if len(law) < MIN_LAW_CHARS or not act_title:      # skip junk / malformed rows
            dropped += 1
            continue

        clean_text = strip_boilerplate(act_title, law)
        doc_id = make_doc_id(act_title, section)

        # If the section already fits in one chunk, keep it as a single "piece" (a list with 1 item)
        # so the loop below behaves identically whether or not splitting actually happened.
        pieces = [clean_text] if len(clean_text) <= MAX_CHUNK_CHARS else splitter.split_text(clean_text)
        for i, piece in enumerate(pieces):                 # enumerate gives (index, value) pairs
            records.append({
                "id": f"{doc_id}_{i}",           # per-chunk id: parent id + child index, e.g. "abc123..._0"
                "act_title": act_title,
                "section": section,
                "parent_doc_id": doc_id,          # every child of the same section shares this value
                "chunk_text": piece,               # the small piece -- this is what gets embedded/searched
                "parent_text": clean_text,         # the full original section -- this is what the LLM sees
            })

    print(f"Chunks: {len(records)} | dropped empty/malformed rows: {dropped}")
    return records

records = load_and_chunk(split="train", num_samples=200)
records[0]   # peek at one record's shape before moving on

README.md:   0%|          | 0.00/615 [00:00<?, ?B/s]

data/indian_law_bare_acts_dataset.parque(…):   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Chunks: 256 | dropped empty/malformed rows: 0


{'id': '63133ba011f631fd_0',
 'act_title': 'Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016',
 'section': '1',
 'parent_doc_id': '63133ba011f631fd',
 'chunk_text': 'The Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016\nChapter I\nPreliminary\n1. Short title, extent and commencement.-\n(1) This Act may be called the Aadhaar (Targeted Delivery of Financial and Other Subsidies, Benefits and Services) Act, 2016.\n(2) It shall extend to the whole of India 2*** and save as otherwise provided in this Act, it shall also apply to any offence or contravention thereunder committed outside India by any person.\n(3) It shall come into force on such date3 as the Central Government may, by notification in the Official Gazette, appoint; and different dates may, be appointed for different provisions of this Act and any reference in any such provision to the commencement of this Act shall be construed as a referen

## 3. Embeddings -- mxbai-embed-large-v1 (dense) + BM25 (sparse)

**The method -- hybrid embeddings:** we compute *two* different representations of each chunk:

- **Dense vectors** (via `mxbai-embed-large-v1`, or OpenAI's `text-embedding-3-large` if
  `EMBEDDING_PROVIDER = "openai"`) capture *semantic meaning* -- "penalty for theft" and "punishment
  for stealing" end up close together in vector space even though they share almost no words.
- **Sparse vectors** (via `fastembed`'s BM25 implementation) capture *exact lexical/keyword overlap* --
  critical for legal text, where a query like "Section 302" or a specific defined term needs to match
  the literal token, not just something semantically similar.

Neither alone is reliable for legal retrieval: dense embeddings can miss an exact section number match,
while pure keyword search misses paraphrased questions. Section 5 fuses both result lists together.

*Example:* for the query "what happens if someone kills another person", dense embeddings will surface
IPC Section 302 ("Punishment for murder") even without the word "kill" appearing verbatim, because the
model has learned that "kills" and "murder" are semantically related. A pure keyword/BM25 search might
rank a section that happens to repeat "person" many times above the actually-relevant one.


In [ ]:
from openai import OpenAI
import numpy as np
from sentence_transformers import SentenceTransformer
from fastembed import TextEmbedding, SparseTextEmbedding   # lightweight local embedding runtime

EMBEDDING_PROVIDER = "openai" # "local" (mxbai via fastembed, free, runs on this machine) or "openai" (API call)
OPENAI_EMBED_MODEL = "text-embedding-3-large"
OPENAI_EMBED_DIM = 1024 # truncated via `dimensions=` to match DENSE_DIM in Section 4

# TextEmbedding / SparseTextEmbedding download the model weights on first use, then run locally --
# this line has network + CPU/GPU cost the first time it executes.
dense_model = TextEmbedding(model_name="mixedbread-ai/mxbai-embed-large-v1")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def embed_dense_local(texts: list[str]) -> list[list[float]]:
    # dense_model.embed(...) returns a *generator* of numpy arrays (memory-efficient for big batches),
    # so we materialize it into a plain list, and convert each numpy array to a plain Python list of
    # floats with .tolist() (numpy arrays aren't JSON/Qdrant-payload friendly on their own).
    return [vec.tolist() for vec in dense_model.embed(texts)]

def embed_dense_openai(texts: list[str]) -> list[list[float]]:
    # One API call embeds the whole batch of texts at once (cheaper & faster than one call per text).
    # `dimensions=` asks OpenAI's Matryoshka-style embedding model to return a shorter vector directly,
    # rather than us truncating a 3072-dim vector ourselves.
    resp = openai_client.embeddings.create(
        model=OPENAI_EMBED_MODEL, input=texts, dimensions=OPENAI_EMBED_DIM,
    )
    return [d.embedding for d in resp.data]   # resp.data is a list of embedding objects, same order as input

def embed_dense(texts: list[str]) -> list[list[float]]:
    # Single switch point -- every other function in this notebook calls embed_dense(), never the
    # provider-specific functions directly, so flipping EMBEDDING_PROVIDER doesn't require code changes elsewhere.
    if EMBEDDING_PROVIDER == "openai":
        return embed_dense_openai(texts)
    return embed_dense_local(texts)

def embed_sparse(texts: list[str]):
    # Returns fastembed SparseEmbedding objects (each has .indices and .values arrays) -- see Section 4/5
    # for how these get converted into Qdrant's SparseVector format.
    return list(sparse_model.embed(texts))

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

## 4. Qdrant -- hybrid collection

**The method:** Qdrant lets a single "collection" (its version of a table) store *multiple named
vectors* per point. We use this to keep the dense vector and the sparse vector for the same chunk
attached to the same point, instead of running two separate databases. `":memory:"` mode means this
Qdrant instance lives entirely in this Python process's RAM -- perfect for a notebook demo, but it
disappears when the runtime restarts (for production you'd point `QdrantClient` at a real host/URL
or a local file path instead).

The `Modifier.IDF` on the sparse vector config tells Qdrant to apply IDF (inverse document frequency)
weighting automatically -- so common words like "the" or "shall" (which appear in almost every legal
section) contribute less to the sparse-search score than rare, distinguishing words like "abetment" or
"habeas corpus".


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, Modifier, PointStruct, SparseVector,
    Filter, FieldCondition, MatchValue,
)

# ":memory:" -> an in-process, RAM-only Qdrant instance (no server to run, no persistence across restarts).
client = QdrantClient(":memory:")

COLLECTION = "indian_laws"
DENSE_DIM = 1024   # must match OPENAI_EMBED_DIM / the dense model's output size, or inserts will fail

client.recreate_collection(              # drops the collection if it exists, then creates it fresh
    collection_name=COLLECTION,
    # vectors_config is a dict of {name: VectorParams} -- this is what makes the collection "hybrid":
    # one named vector space ("dense") using cosine distance for semantic similarity.
    vectors_config={"dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE)},
    # sparse_vectors_config is a *separate* dict for sparse (keyword) vectors -- Qdrant indexes and
    # queries these differently from dense vectors. Modifier.IDF applies automatic term weighting.
    sparse_vectors_config={"sparse": SparseVectorParams(modifier=Modifier.IDF)},  # <- added modifier
)

/tmp/ipykernel_854/2730157134.py:13: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(              # drops the collection if it exists, then creates it fresh


True

In [ ]:
import uuid

def get_uuid(point_id):
    # Qdrant point IDs must be either an unsigned integer or a valid UUID string -- our own ids
    # (e.g. "abc123..._0") aren't valid UUIDs, so uuid5 deterministically maps any string to a UUID.
    # uuid5 (unlike uuid4) is *not* random: the same (namespace, name) input always produces the same
    # UUID, which is exactly what we want for idempotent upserts (re-running this cell won't create duplicates).
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, point_id))

def upsert_records(records, batch_size=32):
    # Process in batches rather than one record at a time -- embedding APIs (and local models) are far
    # more efficient when given many texts per call instead of many single-text calls.
    for i in range(0, len(records), batch_size):
        batch = records[i:i + batch_size]                 # Python slice -- safely stops at list end
        texts = [r["chunk_text"] for r in batch]            # list comprehension: pull just the text field
        dense_vecs = embed_dense(texts)
        sparse_vecs = embed_sparse(texts)

        points = []
        # zip(...) pairs up the three parallel lists element-by-element (they're the same length/order).
        for rec, dense_vec, sparse_vec in zip(batch, dense_vecs, sparse_vecs):
            points.append(PointStruct(
                id=get_uuid(rec["id"]),
                vector={                                     # matches the named-vector schema from Section 4
                    "dense": dense_vec,
                    "sparse": SparseVector(
                        # fastembed's sparse output stores its data as numpy arrays; Qdrant's SparseVector
                        # wants plain Python lists, hence .tolist() again.
                        indices=sparse_vec.indices.tolist(),
                        values=sparse_vec.values.tolist(),
                    ),
                },
                payload={                                     # payload = arbitrary metadata stored alongside the vectors
                    "act_title": rec["act_title"],
                    "section": rec["section"],
                    "parent_doc_id": rec["parent_doc_id"],
                    "chunk_text": rec["chunk_text"],
                    "parent_text": rec["parent_text"],
                },
            ))
        client.upsert(collection_name=COLLECTION, points=points)   # "upsert" = insert or update if id exists
    print(f"Upserted {len(records)} records.")

upsert_records(records)

Upserted 256 records.


## 5. Retrieval -- RRF fusion, MMR, adaptive k, metadata pre-filtering

**Reciprocal Rank Fusion (RRF):** dense (cosine similarity) and sparse (BM25) scores live on
completely different numeric scales, so averaging them directly would be meaningless -- a cosine score
of 0.8 and a BM25 score of 8.0 aren't comparable. RRF sidesteps this by ignoring the raw scores
entirely and fusing based on *rank position* instead: `score = sum(1 / (k + rank))` across every
result list a document appears in. A document ranked #1 in *both* lists ends up scoring higher than one
ranked #1 in only one list.

*Tiny worked example* (k=60): if "Section 302" is rank 1 in the dense list and rank 3 in the sparse
list, its RRF score is `1/(60+1) + 1/(60+3) = 0.01639 + 0.01587 = 0.03226`. A document that's rank 2
in *both* lists scores `1/(60+2) + 1/(60+2) = 0.03226` -- almost identical, showing how RRF rewards
consistent agreement across both retrieval methods rather than one method's outlier top score.

**MMR (Maximal Marginal Relevance):** after fusion we still have `fused_k` (25) candidates, several of
which might be near-duplicate chunks of the *same* section. MMR re-orders the list to balance
"relevant to the query" against "different from what's already been picked" -- so the reranker (an
expensive step) sees a diverse set of candidates instead of 15 near-identical ones.

*Example:* if 4 of the top dense hits are all different child-chunks of the same Section 375, MMR will
pick the single best one early, then downweight the others (they're too similar to what's already
selected) so their retrieval slots go to genuinely different sections instead.

**Metadata pre-filtering:** if the (rewritten) query names a specific Act, `act_filter` narrows the
search to only points where `act_title` matches -- otherwise a section number like "302" could
collide across different acts that happen to share that numbering.


In [ ]:
import time

def build_filter(act_filter):
    # Returns None (no filtering) if act_filter wasn't provided, otherwise builds a Qdrant Filter
    # that restricts search to points whose payload["act_title"] exactly equals act_filter.
    if not act_filter:
        return None
    return Filter(must=[FieldCondition(key="act_title", match=MatchValue(value=act_filter))])

def reciprocal_rank_fusion(result_lists, k=60):
    # k=60 is the standard RRF damping constant from the original paper -- large enough that rank 1
    # vs rank 2 doesn't dominate the score, so multiple *moderate* ranks across lists can outweigh one
    # single top rank.
    scores, payloads, vectors = {}, {}, {}
    for results in result_lists:                              # results = one ranked list (dense OR sparse hits)
        for rank, hit in enumerate(results):                   # rank starts at 0 here, hence "+ rank + 1" below
            scores[hit.id] = scores.get(hit.id, 0.0) + 1.0 / (k + rank + 1)
            payloads[hit.id] = hit.payload
            vectors[hit.id] = hit.vector["dense"]   # named-vector dict: {"dense": [...], "sparse": SparseVector(...)}
    # sorted(..., key=lambda x: x[1], reverse=True) sorts (id, score) tuples by score, highest first.
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {"id": doc_id, "score": score, "payload": payloads[doc_id], "dense_vector": vectors[doc_id]}
        for doc_id, score in ranked
    ]

def hybrid_search(query: str, top_k: int = 25, act_filter=None):
    t0 = time.time()                                # time.time() -- wall-clock timestamp, used purely for the
    dense_vec = embed_dense([query])[0]              # timing prints below (helpful when debugging latency)
    print(f"  embed query (dense): {time.time()-t0:.2f}s")

    t0 = time.time()
    sparse_vec = embed_sparse([query])[0]
    print(f"  embed query (sparse): {time.time()-t0:.2f}s")

    query_filter = build_filter(act_filter)

    t0 = time.time()
    # client.query_points(...) is Qdrant's search call. `using="dense"` tells it which named vector
    # space to search against; `with_vectors=True` asks Qdrant to also return the stored vectors
    # (needed later for MMR, which compares candidate vectors to each other).
    dense_hits = client.query_points(
        collection_name=COLLECTION, query=dense_vec, using="dense",
        limit=top_k, query_filter=query_filter, with_vectors=True,
    ).points
    print(f"  qdrant dense search: {time.time()-t0:.2f}s")

    t0 = time.time()
    sparse_hits = client.query_points(
        collection_name=COLLECTION,
        query=SparseVector(indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()),
        using="sparse", limit=top_k, query_filter=query_filter, with_vectors=True,
    ).points
    print(f"  qdrant sparse search: {time.time()-t0:.2f}s")

    return reciprocal_rank_fusion([dense_hits, sparse_hits])

def mmr_select(query_vec, candidates, candidate_vecs, top_n=15, lambda_param=0.5):
    if len(candidates) == 0:
        return []

    query_vec = np.array(query_vec)
    candidate_vecs = np.array(candidate_vecs)   # <- coerces list-of-lists into a real 2D array

    # Vectorized cosine similarity of every candidate against the query in one shot:
    # (N, D) @ (D,) -> (N,) dot products, divided by the product of norms. The "+ 1e-9" avoids a
    # divide-by-zero if any vector norm is exactly 0.
    sims_to_query = candidate_vecs @ query_vec / (
        np.linalg.norm(candidate_vecs, axis=1) * np.linalg.norm(query_vec) + 1e-9
    )
    selected_idx, remaining = [], list(range(len(candidates)))
    # Always start by greedily picking the single most query-relevant candidate.
    selected_idx.append(remaining.pop(int(np.argmax(sims_to_query[remaining]))))

    while remaining and len(selected_idx) < top_n:
        mmr_scores = []
        for i in remaining:
            # For candidate i, find how similar it is to the *most similar* item already selected
            # (max(...) over a generator expression) -- this is the "redundancy penalty" term.
            sim_to_selected = max(
                np.dot(candidate_vecs[i], candidate_vecs[j])
                / (np.linalg.norm(candidate_vecs[i]) * np.linalg.norm(candidate_vecs[j]) + 1e-9)
                for j in selected_idx
            )
            # The MMR formula: balance relevance to the query against similarity to what's already
            # picked. lambda_param=0.5 weighs both terms equally; closer to 1.0 favors pure relevance,
            # closer to 0.0 favors pure diversity.
            mmr_scores.append(lambda_param * sims_to_query[i] - (1 - lambda_param) * sim_to_selected)
        chosen = remaining[int(np.argmax(mmr_scores))]   # np.argmax returns the *index within mmr_scores*
        selected_idx.append(chosen)
        remaining.remove(chosen)
    return [candidates[i] for i in selected_idx]

def retrieve(query: str, act_filter=None, fused_k: int = 25, mmr_k: int = 15):
    print(f"retrieve() called for: {query!r}")   # !r -> repr(), so the string prints with quotes/escapes visible
    fused = hybrid_search(query, top_k=fused_k, act_filter=act_filter)
    if not fused:
        return []
    t0 = time.time()
    query_vec = embed_dense([query])[0]
    cand_vecs = [c["dense_vector"] for c in fused]
    result = mmr_select(query_vec, fused, cand_vecs, top_n=mmr_k)
    print(f"  mmr_select: {time.time()-t0:.2f}s")
    return result

## 6. Reranker -- BGE-reranker-v2-m3

**The method:** a cross-encoder reranker looks at the query and *one* candidate chunk together (as a
single input pair) and outputs a relevance score for that specific pair -- unlike the embedding models
above, which score the query and each chunk *independently* and compare vectors afterwards. Reading
query and candidate jointly is far more accurate, but far more expensive (it can't be pre-computed and
cached like embeddings can), which is exactly why it only runs on the small MMR-filtered shortlist
(15 candidates) rather than the whole collection.

*Example:* for the query "murder punishment", a bi-encoder (embedding) might rate "Section 302" and
"Section 300" (culpable homicide, a related-but-distinct offence) similarly close in vector space. The
cross-encoder reads both texts *together* with the query and can pick up on the finer distinction that
Section 302 states the actual punishment, ranking it higher.


In [ ]:
from sentence_transformers import CrossEncoder

# CrossEncoder downloads/loads the BGE reranker model -- this happens once, the object is reused below.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query: str, candidates: list, top_n: int = 6):
    if not candidates:
        return []
    # Build (query, candidate_text) pairs -- the cross-encoder scores each pair jointly.
    pairs = [[query, c["payload"]["chunk_text"]] for c in candidates]
    scores = reranker.predict(pairs)  # raw logits; not 0-1 normalized like FlagReranker's normalize=True
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)          # numpy float32 -> plain Python float, safer for JSON/printing
    # Sort candidates by their new rerank_score, highest first, then keep only the top_n.
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_n]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

## 7. Generator router -- OpenAI / Groq / Gemini

**The method:** `generate()` is a single function with one job -- take a prompt and a provider name,
and return the model's text response -- regardless of which LLM vendor is actually being called. Every
node in the LangGraph pipeline (Section 10) calls this one function, so switching the whole pipeline
from OpenAI to Groq or Gemini is a one-line change (`provider="groq"`) rather than a rewrite.

Swap model names below for whatever's current in each provider's catalog at the time you run this.


In [ ]:
from openai import OpenAI
from groq import Groq
import google.genai as genai

openai_client = OpenAI(api_key=OPENAI_API_KEY)
# groq_client = Groq(api_key=GROQ_API_KEY)        # uncomment together with GROQ_API_KEY above to enable
# genai.configure(api_key=GEMINI_API_KEY)          # uncomment together with GEMINI_API_KEY above to enable

# Central place to pin default model names per provider -- change here, not at each call site.
DEFAULT_MODELS = {
    "openai": "gpt-4o-mini",
    "groq": "llama-3.3-70b-versatile",
    "gemini": "gemini-2.5-flash",
}

def generate(prompt: str, provider: str = "openai", model: Optional[str] = None) -> str:
    model = model or DEFAULT_MODELS[provider]   # `model or DEFAULT` -- use caller's model if given, else the default

    if provider == "openai":
        # Chat Completions API: messages is a list of {"role", "content"} turns; here it's a single
        # user turn since we build the full prompt (including context/history) as one string above.
        resp = openai_client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content   # .choices is a list (supports n>1 completions); we take the first

    if provider == "groq":
        # Groq's SDK mirrors the OpenAI chat-completions shape, so the call looks almost identical.
        resp = groq_client.chat.completions.create(
            model=model, messages=[{"role": "user", "content": prompt}],
        )
        return resp.choices[0].message.content

    if provider == "gemini":
        # Gemini's SDK shape differs: instantiate a GenerativeModel, then call generate_content directly
        # with the raw prompt string (no messages list).
        resp = genai.GenerativeModel(model).generate_content(prompt)
        return resp.text

    raise ValueError(f"Unknown provider: {provider}")   # fail loudly on a typo'd provider name

## 8. Web search fallback -- Serper

**The method:** if the top reranked chunk's confidence score is below `CONFIDENCE_THRESHOLD`, the
pipeline assumes the local 200-row sample dataset simply doesn't contain the answer, and falls back to
a live web search instead of letting the LLM guess from weak context. The Serper query is deliberately
scoped with `site:indiacode.nic.in OR site:indiankanoon.org` so results come from authoritative Indian
legal sources rather than arbitrary blogs or forums.

This is a *safety valve*, not the primary path -- in the full dataset (not just 200 sampled rows) most
queries should be answered from the vector store, and the web fallback should trigger rarely.


In [ ]:
import requests

CONFIDENCE_THRESHOLD = 0.35   # rerank_score below this triggers node_web_search instead of node_context_assembly

def serper_search(query: str, num: int = 5) -> list[dict]:
    url = "https://google.serper.dev/search"
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    # site:a OR site:b restricts Google's results to just those two domains.
    payload = {"q": f"{query} site:indiacode.nic.in OR site:indiankanoon.org", "num": num}
    resp = requests.post(url, headers=headers, json=payload, timeout=15)   # timeout guards against a hung request
    resp.raise_for_status()   # raises an exception if Serper returned a 4xx/5xx status code
    data = resp.json()
    # Serper's organic results carry more fields than we need -- pull out just title/snippet/link.
    return [
        {"title": item.get("title"), "snippet": item.get("snippet"), "link": item.get("link")}
        for item in data.get("organic", [])   # .get(..., []) -> empty list if "organic" key is missing entirely
    ]

## 9. Memory -- SQLite

**The method:** every conversation turn (query + rewritten query + answer + citations) is appended to
a per-`session_id` JSON blob stored in a local SQLite file. Before every new turn, `node_rewrite_query`
(Section 10) pulls the last few turns for that session and asks the LLM to rewrite a follow-up
question into a fully standalone one -- this is what lets "what about for minors?" become "What is the
punishment for murder under IPC for minors?" using the prior turn's context.

SQLite is used here (rather than, say, a dict in memory) specifically so history *survives* if the
Python process restarts -- the trade-off is every read/write opens a small file-backed connection,
which is why Section 12 gives you a way to peek at what's actually stored there.


In [ ]:
import sqlite3
import json as _json   # aliased to avoid shadowing the `json` module imported in Section 1

DB_PATH = "indian_laws_rag_memory.db"

def _get_conn():
    conn = sqlite3.connect(DB_PATH)   # opens (or creates, if missing) the local SQLite file
    conn.execute(
        """CREATE TABLE IF NOT EXISTS sessions (
            session_id TEXT PRIMARY KEY,
            turns TEXT NOT NULL
        )"""
    )   # IF NOT EXISTS -- safe to run on every connection without wiping existing data
    return conn

def get_chat_history(session_id: str, limit: int = 5) -> list[dict]:
    conn = _get_conn()
    # Parameterized query ("?" placeholder + tuple of values) -- avoids SQL injection and correctly
    # quotes the session_id value regardless of its contents.
    row = conn.execute(
        "SELECT turns FROM sessions WHERE session_id = ?", (session_id,)
    ).fetchone()   # fetchone() -> a single row tuple, or None if no match
    conn.close()
    if not row:
        return []
    turns = _json.loads(row[0])   # row[0] is the "turns" column -- a JSON string -- parse it back to a list
    return turns[-limit:]          # Python negative-index slice: the last `limit` items

def append_turn(session_id: str, turn: dict):
    conn = _get_conn()
    row = conn.execute(
        "SELECT turns FROM sessions WHERE session_id = ?", (session_id,)
    ).fetchone()
    turns = _json.loads(row[0]) if row else []   # existing history, or a fresh empty list for a new session
    turns.append(turn)
    # "ON CONFLICT(session_id) DO UPDATE" -- SQLite's upsert syntax: insert a new row normally, but if
    # session_id already exists (it's the PRIMARY KEY), update its "turns" column instead of erroring.
    conn.execute(
        "INSERT INTO sessions (session_id, turns) VALUES (?, ?) "
        "ON CONFLICT(session_id) DO UPDATE SET turns = excluded.turns",
        (session_id, _json.dumps(turns)),
    )
    conn.commit()   # writes are not durable until commit() is called
    conn.close()

## 10. LangGraph pipeline

**The method:** LangGraph models the pipeline as a graph of nodes, where every node is a plain
function `(state) -> state` that reads and updates a shared state dict, and edges define what runs
next. This is the same idea as a flowchart: instead of one long function with lots of `if/else`
branches, each step is isolated, testable on its own, and the *routing logic* (which step follows
which) is declared separately from the step logic itself.

**`RAGState` (a `TypedDict`)** is the schema of that shared dict -- it doesn't enforce types at
runtime, but it documents exactly what every node can read/write, and gives editor autocomplete.

**Conditional edges:** `route_confidence` is a function that inspects the state *after* reranking and
returns a string naming which node runs next (`"web_search"` or `"context_assembly"`) -- this is how
the confidence-gate branch from Section 8 is wired into the graph.

Full path: load memory -> rewrite query -> decompose -> retrieve (RRF + MMR) -> rerank -> confidence
gate -> (web search | context assembly) -> generate + cite -> persist memory.


In [ ]:
from langgraph.graph import StateGraph, END

class RAGState(TypedDict):
    session_id: str
    raw_query: str            # the user's original, possibly-a-follow-up question
    rewritten_query: str      # standalone version of raw_query, using chat history if needed
    sub_queries: list         # rewritten_query decomposed into 1+ standalone sub-questions
    chat_history: list        # last few turns for this session_id, loaded from SQLite
    act_filter: Optional[str]
    retrieved_chunks: list    # output of Section 5's retrieve() across all sub_queries
    reranked_chunks: list     # output of Section 6's rerank()
    confidence_score: float   # top reranked chunk's score -- drives route_confidence
    source_path: str          # "vector_store" or "web_search" -- which branch actually produced the context
    context: str              # the assembled text block handed to the LLM as grounding
    answer: str
    citations: list
    provider: str             # which LLM provider (Section 7) this run should use

def node_load_history(state: RAGState) -> RAGState:
    # Every node follows the same shape: take state in, mutate/add keys, return state.
    state["chat_history"] = get_chat_history(state["session_id"])
    return state

def node_rewrite_query(state: RAGState) -> RAGState:
    if not state["chat_history"]:
        # First turn in a session -- nothing to rewrite against, so pass the raw query through unchanged.
        state["rewritten_query"] = state["raw_query"]
        return state
    # Flatten prior turns into a readable "Q: ... A: ..." transcript for the LLM prompt.
    history_text = "\n".join(f"Q: {t['query']}\nA: {t['answer']}" for t in state["chat_history"])
    prompt = (
        f"Conversation history:\n{history_text}\n\n"
        f"Rewrite the follow-up question into a standalone, fully-specified question. "
        f"Output only the rewritten question, nothing else.\n"
        f"Follow-up: {state['raw_query']}"
    )
    state["rewritten_query"] = generate(prompt, provider=state.get("provider", "openai")).strip()
    return state

def node_decompose(state: RAGState) -> RAGState:
    # Ask the LLM to return a JSON list of sub-questions -- one query might bundle two legal questions
    # together (e.g. "what's the punishment for theft AND for criminal breach of trust?").
    prompt = (
        "If this legal question has multiple distinct parts, split it into a JSON list of "
        "standalone sub-questions. If it is a single question, return a JSON list with just "
        "that one question. Return ONLY a JSON list of strings, no other text.\n"
        f"Question: {state['rewritten_query']}"
    )
    raw = generate(prompt, provider=state.get("provider", "openai"))
    try:
        sub_qs = json.loads(raw)
        # Defensive check: even though we asked for a JSON list of strings, LLM output isn't
        # guaranteed -- assert both that it parsed to a list AND every element is actually a string.
        assert isinstance(sub_qs, list) and all(isinstance(x, str) for x in sub_qs)
    except Exception:
        # Any parse/format failure -> safe fallback: treat the whole rewritten query as one sub-question.
        sub_qs = [state["rewritten_query"]]
    state["sub_queries"] = sub_qs
    return state

def node_retrieve(state: RAGState) -> RAGState:
    print(f"sub_queries: {state['sub_queries']}")
    seen = {}   # dedupe chunks retrieved by multiple sub-queries, keeping the highest score per chunk id
    for sq in state["sub_queries"]:
        for c in retrieve(sq, act_filter=state.get("act_filter")):
            if c["id"] not in seen or c["score"] > seen[c["id"]]["score"]:
                seen[c["id"]] = c
    state["retrieved_chunks"] = list(seen.values())   # dict values -> plain list for downstream nodes
    return state

def node_rerank(state: RAGState) -> RAGState:
    reranked = rerank(state["rewritten_query"], state["retrieved_chunks"], top_n=6)
    state["reranked_chunks"] = reranked
    # Confidence = top result's rerank score, or 0.0 if nothing came back at all (empty list is falsy).
    state["confidence_score"] = reranked[0]["rerank_score"] if reranked else 0.0
    return state

def route_confidence(state: RAGState) -> Literal["web_search", "context_assembly"]:
    # This function's *return value* (a string) is what LangGraph uses to decide which node runs next --
    # it must exactly match one of the keys in the conditional-edges mapping registered below.
    return "web_search" if state["confidence_score"] < CONFIDENCE_THRESHOLD else "context_assembly"

def node_web_search(state: RAGState) -> RAGState:
    results = serper_search(state["rewritten_query"])
    # Build one big context string of "[Web: title]\nsnippet\nSource: link" blocks, joined with blank lines.
    state["context"] = "\n\n".join(
        f"[Web: {r['title']}]\n{r['snippet']}\nSource: {r['link']}" for r in results
    )
    state["source_path"] = "web_search"
    state["citations"] = [{"source": r["link"], "title": r["title"]} for r in results]
    return state

def node_context_assembly(state: RAGState) -> RAGState:
    parts, citations = [], []
    for c in state["reranked_chunks"]:
        p = c["payload"]
        # Note: parent_text (the FULL section), not chunk_text (the small retrieved piece) --
        # this is the "small-to-big" retrieval pattern from Section 2 paying off at generation time.
        parts.append(f"[Act: {p['act_title']} | Section: {p['section']}]\n{p['parent_text']}")
        citations.append({"act_title": p["act_title"], "section": p["section"]})
    state["context"] = "\n\n".join(parts)
    state["source_path"] = "vector_store"
    state["citations"] = citations
    return state

def node_generate(state: RAGState) -> RAGState:
    # The prompt explicitly instructs the model to cite Act+Section per claim, and to admit when the
    # context doesn't contain the answer -- reducing (not eliminating) hallucination risk.
    prompt = (
        "You are a legal assistant answering questions about Indian law. "
        "Use ONLY the context below. Cite the Act and Section for every claim. "
        "If the answer isn't in the context, say so explicitly rather than guessing.\n\n"
        f"Context:\n{state['context']}\n\n"
        f"Question: {state['rewritten_query']}\n\nAnswer:"
    )
    state["answer"] = generate(prompt, provider=state.get("provider", "openai"))
    return state

def node_persist_memory(state: RAGState) -> RAGState:
    # This is the write that Section 12's SQLite inspection cell will read back.
    append_turn(state["session_id"], {
        "query": state["raw_query"],
        "rewritten_query": state["rewritten_query"],
        "answer": state["answer"],
        "source_path": state["source_path"],
        "citations": state["citations"],
    })
    return state

# --- Assemble the graph ---
graph = StateGraph(RAGState)             # StateGraph is generic over the state schema (RAGState above)
graph.add_node("load_history", node_load_history)
graph.add_node("rewrite_query", node_rewrite_query)
graph.add_node("decompose", node_decompose)
graph.add_node("retrieve", node_retrieve)
graph.add_node("rerank", node_rerank)
graph.add_node("web_search", node_web_search)
graph.add_node("context_assembly", node_context_assembly)
graph.add_node("generate", node_generate)
graph.add_node("persist_memory", node_persist_memory)

graph.set_entry_point("load_history")             # where every graph.invoke() run starts
graph.add_edge("load_history", "rewrite_query")   # unconditional edge: always go A -> B
graph.add_edge("rewrite_query", "decompose")
graph.add_edge("decompose", "retrieve")
graph.add_edge("retrieve", "rerank")
# Conditional edge: after "rerank", call route_confidence(state); whatever string it returns is looked
# up in this mapping to decide the actual next node.
graph.add_conditional_edges("rerank", route_confidence, {
    "web_search": "web_search",
    "context_assembly": "context_assembly",
})
graph.add_edge("web_search", "generate")          # both branches converge back to "generate"
graph.add_edge("context_assembly", "generate")
graph.add_edge("generate", "persist_memory")
graph.add_edge("persist_memory", END)             # END is LangGraph's sentinel for "graph run finished"

app = graph.compile()   # compiles the graph definition into a runnable object -- app.invoke(...) executes it

## 11. Demo run

**Setup:** upload a CSV of sample queries (e.g. via Colab's file upload icon, or `files.upload()`) named
`sample_queries.csv` with at minimum a `query` column. Optional columns:
- `session_id` -- groups queries into conversations; rows sharing a session_id can exercise the
  query-rewriter/memory logic as follow-ups. Defaults to an auto-generated id per row if omitted.
- `act_filter` -- restricts retrieval to one Act (see Section 5). Leave blank/omit for no filter.

If no CSV is present, the cell below falls back to a small built-in sample set so the notebook still
runs end-to-end without any upload.


### From Vector Store

In [ ]:
import pandas as pd
import os as _os

CSV_PATH = "Laws Dataset Sample queries.csv"

# Small built-in fallback so this cell (and the rest of the demo) still runs even before you've
# uploaded your own CSV -- exercises both branches of the confidence gate and the memory/follow-up path.
FALLBACK_QUERIES = pd.DataFrame([
    {"session_id": "demo-murder",  "query": "What is the punishment for murder under IPC?", "act_filter": ""},
    {"session_id": "demo-murder",  "query": "what about for minors?", "act_filter": ""},
    {"session_id": "demo-theft",   "query": "What is the punishment for theft under IPC?", "act_filter": ""},
    {"session_id": "demo-obscure", "query": "What does the Constitution say about a right to privacy in space travel?", "act_filter": ""},
])

if _os.path.exists(CSV_PATH):
    queries_df = pd.read_csv(CSV_PATH)
    # Fill in optional columns if the uploaded CSV omits them, so downstream code never KeyErrors.
    if "session_id" not in queries_df.columns:
        queries_df["session_id"] = [f"csv-session-{i}" for i in range(len(queries_df))]
    if "act_filter" not in queries_df.columns:
        queries_df["act_filter"] = ""
    print(f"Loaded {len(queries_df)} queries from {CSV_PATH}")
else:
    queries_df = FALLBACK_QUERIES
    print(f"No {CSV_PATH} found -- using {len(queries_df)} built-in fallback queries. "
          f"Upload a CSV with a 'query' column (and optional 'session_id', 'act_filter') to test your own.")

queries_df.head(5)

No Laws Dataset Sample queries.csv found -- using 4 built-in fallback queries. Upload a CSV with a 'query' column (and optional 'session_id', 'act_filter') to test your own.


,session_id,query,act_filter
0,demo-murder,What is the punishment for murder under IPC?,
1,demo-murder,what about for minors?,
2,demo-theft,What is the punishment for theft under IPC?,
3,demo-obscure,What does the Constitution say about a right t...,


In [ ]:
# Run every row in queries_df through the compiled LangGraph app, collecting results as we go.
demo_results = []

# for _, row in queries_df.iterrows(): # iterrows() yields (index, row) -- row behaves like a dict/Series
for _, row in queries_df.head(5).iterrows():
    act_filter = row["act_filter"] if pd.notna(row["act_filter"]) and row["act_filter"] != "" else None
    print(f"\n{'='*80}\nQuery: {row['query']}  (session={row['session_id']})\n{'='*80}")

    result = app.invoke({
        "session_id": row["session_id"],
        "raw_query": row["query"],
        "act_filter": act_filter,
        "provider": "openai",
    })

    demo_results.append({
        "session_id": row["session_id"],
        "query": row["query"],
        "rewritten_query": result["rewritten_query"],
        "source_path": result["source_path"],
        "confidence": round(result["confidence_score"], 3),
        "answer": result["answer"],
        "citations": result["citations"],
    })

    print("Source path:", result["source_path"])
    print("Confidence:", round(result["confidence_score"], 3))
    print("Answer:\n", result["answer"])


Query: What is the punishment for murder under IPC?  (session=demo-murder)
sub_queries: ['What is the punishment for murder under IPC?']
retrieve() called for: 'What is the punishment for murder under IPC?'
  embed query (dense): 0.39s
  embed query (sparse): 0.00s
  qdrant dense search: 0.01s
  qdrant sparse search: 0.01s
  mmr_select: 0.38s
Source path: web_search
Confidence: 0.004
Answer:
 The punishment for murder under the Indian Penal Code (IPC) is outlined in Section 302. It states that "Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine." (IPC, Section 302).

Query: what about for minors?  (session=demo-murder)
sub_queries: ['What is the punishment for murder under the Indian Penal Code (IPC)?', 'How does the punishment for murder differ when the offender is a minor under the IPC?']
retrieve() called for: 'What is the punishment for murder under the Indian Penal Code (IPC)?'
  embed query (dense): 0.40s
  embed query

In [ ]:
# Summary table across every query run above -- easiest way to eyeball which queries hit the vector
# store vs fell back to web search, and to spot-check confidence scores at a glance.
results_df = pd.DataFrame(demo_results)[["session_id", "query", "source_path", "confidence", "rewritten_query"]]
results_df

,session_id,query,source_path,confidence,rewritten_query
0,demo-murder,What is the punishment for murder under IPC?,web_search,0.004,What is the punishment for murder under IPC?
1,demo-murder,what about for minors?,web_search,0.002,What is the punishment for murder under the In...
2,demo-theft,What is the punishment for theft under IPC?,web_search,0.067,What is the punishment for theft under IPC?
3,demo-obscure,What does the Constitution say about a right t...,web_search,0.001,What does the Constitution say about a right t...


**Follow-up / memory check:** the `demo-murder` session above includes two turns on purpose --
"What is the punishment for murder under IPC?" followed by "what about for minors?". The second call's
`rewritten_query` should read as a standalone question (something like "What is the punishment for
murder under IPC for minors?") even though the raw query alone has no mention of murder or IPC --
proof that `node_rewrite_query` correctly pulled the prior turn from SQLite memory (Section 9) and used
it to disambiguate the follow-up.


### Fallback

In [ ]:
import pandas as pd
import os as _os

# Small built-in fallback so this cell (and the rest of the demo) still runs even before you've
# uploaded your own CSV -- exercises both branches of the confidence gate and the memory/follow-up path.
FALLBACK_QUERIES = pd.DataFrame([
    {"session_id": "demo-murder",  "query": "What is the punishment for murder under IPC?", "act_filter": ""},
    {"session_id": "demo-murder",  "query": "what about for minors?", "act_filter": ""},
    {"session_id": "demo-theft",   "query": "What is the punishment for theft under IPC?", "act_filter": ""},
    {"session_id": "demo-obscure", "query": "What does the Constitution say about a right to privacy in space travel?", "act_filter": ""},
])

queries_df = FALLBACK_QUERIES

queries_df

,session_id,query,act_filter
0,demo-murder,What is the punishment for murder under IPC?,
1,demo-murder,what about for minors?,
2,demo-theft,What is the punishment for theft under IPC?,
3,demo-obscure,What does the Constitution say about a right t...,


In [ ]:
# Run every row in queries_df through the compiled LangGraph app, collecting results as we go.
demo_results = []

# for _, row in queries_df.iterrows(): # iterrows() yields (index, row) -- row behaves like a dict/Series
for _, row in queries_df.head(5).iterrows():
    act_filter = row["act_filter"] if pd.notna(row["act_filter"]) and row["act_filter"] != "" else None
    print(f"\n{'='*80}\nQuery: {row['query']}  (session={row['session_id']})\n{'='*80}")

    result = app.invoke({
        "session_id": row["session_id"],
        "raw_query": row["query"],
        "act_filter": act_filter,
        "provider": "openai",
    })

    demo_results.append({
        "session_id": row["session_id"],
        "query": row["query"],
        "rewritten_query": result["rewritten_query"],
        "source_path": result["source_path"],
        "confidence": round(result["confidence_score"], 3),
        "answer": result["answer"],
        "citations": result["citations"],
    })

    print("Source path:", result["source_path"])
    print("Confidence:", round(result["confidence_score"], 3))
    print("Answer:\n", result["answer"])


Query: What is the punishment for murder under IPC?  (session=demo-murder)
sub_queries: ['What is the punishment for murder under the Indian Penal Code (IPC)?', 'What are the specific provisions regarding punishment for minors under the Indian Penal Code (IPC)?', 'How does the Juvenile Justice Act affect the punishment for murder by minors in India?']
retrieve() called for: 'What is the punishment for murder under the Indian Penal Code (IPC)?'
  embed query (dense): 0.42s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.00s
  mmr_select: 0.44s
retrieve() called for: 'What are the specific provisions regarding punishment for minors under the Indian Penal Code (IPC)?'
  embed query (dense): 0.38s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.01s
  mmr_select: 0.58s
retrieve() called for: 'How does the Juvenile Justice Act affect the punishment for murder by minors in India?'
  embed query (dense): 0.33s
  embed q

In [ ]:
murder_session_rows = [r for r in demo_results if r["session_id"] == "demo-murder"]
for r in murder_session_rows:
    print(f"Raw query -> rewritten: {r['query']!r} -> {r['rewritten_query']!r}")

Raw query -> rewritten: 'What is the punishment for murder under IPC?' -> 'What is the punishment for murder under the Indian Penal Code (IPC) if the offender is a minor?'
Raw query -> rewritten: 'what about for minors?' -> 'What is the punishment for murder under the Indian Penal Code (IPC) when the offender is a minor?'


## 12. Inspect the datastore

After running the demo queries above, it's worth looking directly at what actually got written to each
store -- both to sanity-check the pipeline and as a debugging habit for when retrieval or memory
behaves unexpectedly.

**Qdrant side:** `client.get_collection()` reports point/vector counts for the collection created in
Section 4. `client.scroll()` pages through raw stored points (payload + optionally vectors) without
running a similarity search -- useful for confirming chunk boundaries, payload fields, and that
small-to-big's `parent_doc_id` grouping looks right.

**SQLite side:** reading straight from `indian_laws_rag_memory.db` confirms that `node_persist_memory`
(Section 10) is actually appending turns per session, and that `node_rewrite_query` had real history
available to read from on the second demo-murder turn.


In [ ]:
# --- Qdrant: collection-level stats ---
collection_info = client.get_collection(collection_name=COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"Points count: {collection_info.points_count}")
print(f"Vectors config: {collection_info.config.params.vectors}")
print(f"Sparse vectors config: {collection_info.config.params.sparse_vectors}")

Collection: indian_laws
Points count: 256
Vectors config: {'dense': VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}
Sparse vectors config: {'sparse': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}


In [ ]:
# --- Qdrant: sample a handful of raw stored points ---
# client.scroll(...) pages through stored points directly (no query vector / similarity search involved),
# returning (points, next_page_offset). We only need the first page here for a quick look.
sample_points, _next_offset = client.scroll(
    collection_name=COLLECTION,
    limit=5,
    with_payload=True,
    with_vectors=False,   # vectors are large and not useful to eyeball -- payload is what we care about here
)

for point in sample_points:
    p = point.payload
    print(f"id={point.id}")
    print(f"  act_title:     {p['act_title']}")
    print(f"  section:       {p['section']}")
    print(f"  parent_doc_id: {p['parent_doc_id']}")
    print(f"  chunk_text:    {p['chunk_text'][:150]}...")   # truncate for readability
    print()

id=0057ca67-f1b9-5aa4-b996-cbc95ab93665
  act_title:     Academy of Scientific and Innovative Research, 2011
  section:       30
  parent_doc_id: fa2b6b3a0a23b7c5
  chunk_text:    The Academy of Scientific and Innovative Research, 2011
30. Ordinances.-
(1) Subject to the provisions of this Act and the Statutes, the Ordinances of...

id=01c1335d-393c-59ad-a5a3-1e02ae3e2e7b
  act_title:     Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016
  section:       2
  parent_doc_id: 9019647a26724c99
  chunk_text:    (e) "Authority" means the Unique Identification Authority of India established under sub-section (1) of section 11;
(f) "benefit" means any advantage,...

id=045a9163-5c2e-5559-99f8-fb2640ba3c00
  act_title:     Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits and Services) Act, 2016
  section:       26
  parent_doc_id: 8ad9c7eaf9f457f8
  chunk_text:    The Aadhaar (Targeted Delivery of Financial and other Subsidies, Be

In [ ]:
# --- SQLite: dump every session's stored turn history ---
conn = _get_conn()
all_sessions = conn.execute("SELECT session_id, turns FROM sessions").fetchall()
conn.close()

print(f"Sessions stored: {len(all_sessions)}\n")
for session_id, turns_json in all_sessions:
    turns = _json.loads(turns_json)
    print(f"Session: {session_id}  ({len(turns)} turn(s))")
    for i, t in enumerate(turns):
        print(t)
    print()

Sessions stored: 3

Session: demo-murder  (4 turn(s))
{'query': 'What is the punishment for murder under IPC?', 'rewritten_query': 'What is the punishment for murder under IPC?', 'answer': 'The punishment for murder under the Indian Penal Code (IPC) is outlined in Section 302. It states that "Whoever commits murder shall be punished with death, or imprisonment for life, and shall also be liable to fine." (IPC, Section 302).', 'source_path': 'web_search', 'citations': [{'source': 'https://indiankanoon.org/doc/626019/', 'title': 'Section 300 in The Indian Penal Code, 1860'}, {'source': 'https://www.indiacode.nic.in/show-data?actid=AC_CEN_5_23_00037_186045_1523266765688&orderno=338', 'title': 'Section 302'}, {'source': 'https://indiankanoon.org/doc/1560742/', 'title': 'Section 302 in The Indian Penal Code, 1860'}, {'source': 'https://indiankanoon.org/search/?formInput=punishment%20for%20murder%3F', 'title': 'punishment for murder?'}, {'source': 'https://indiankanoon.org/doc/305371/', 'tit

## 13. Evaluation -- DeepEval

**Two layers of evaluation:**

1. **Retrieval quality** -- `recall@k` (did the correct section appear anywhere in the top-k
   retrieved/reranked results?) and `MRR`, Mean Reciprocal Rank (how *highly* ranked was it -- `1/rank`,
   so a correct hit at rank 1 scores 1.0, at rank 2 scores 0.5, at rank 4 scores 0.25, and so on).
   These measure the retrieval+rerank pipeline (Sections 5-6) in isolation, independent of the LLM.

2. **Generation quality** -- DeepEval's `GEval` lets you define a *custom* LLM-graded metric from a
   plain-English rubric (here: "Citation Correctness" -- checks that every claim in the answer is
   actually backed by a citation present in the retrieved context). `AnswerRelevancyMetric` and
   `FaithfulnessMetric` are DeepEval's built-in metrics for "does the answer address the question" and
   "does the answer avoid contradicting/inventing beyond the provided context", respectively.

**The gold set:** both layers are scored against `GOLD_SET` -- 20 hand-labeled `(query, correct
(act_title, section) pairs)` examples. Every example is grounded in the actual first 200 rows loaded by
`load_and_chunk(num_samples=200)` in Section 2 (i.e. every gold `(act_title, section)` pair below was
verified to exist in that 200-row slice before being added here), so the gold set only tests questions
the pipeline could plausibly answer from what's actually in the vector store. The 20 questions are
spread across all 10 Acts that fall inside the first 200 rows -- Aadhaar Act 2016 (rows 1-59), Abducted
Persons Act 1955, Absorbed Areas Act 1954, Academy of Scientific and Innovative Research Act 2011,
Acquired Territories (Merger) Act 1960, Acquisition of Certain Area at Ayodhya Act 1993, Actuaries Act
2006, Additional Duties of Excise (Textiles) Act 1978, Administrative Tribunals (Amendment) Act 1986,
and Administrative Tribunals Act 1985 -- instead of only covering Aadhaar, so both retrieval and
generation metrics reflect the pipeline's behaviour across acts, not just one.


In [ ]:
# --- Gold test set (20 examples, all grounded in the first 200 rows of the dataset) ---
#
# Each entry is (query, [(act_title, section), ...]) -- gold_sections is a list because a question
# can legitimately map to more than one correct section, though every example below maps to exactly one.
#
# act_title strings are copied verbatim from the dataset (mratanusarkar/Indian-Laws, rows 0-199) so
# they match exactly what `load_and_chunk` stores in each record's payload.

AADHAAR_ACT = (
    "Aadhaar (Targeted Delivery of Financial and other Subsidies, "
    "Benefits and Services) Act, 2016"
)
ABDUCTED_PERSONS_ACT = "Abducted Persons (Recovery and Restoration) Continuance Act, 1955"
ABSORBED_AREAS_ACT = "Absorbed Areas (Laws) Act, 1954"
ACADEMY_ACT = "Academy of Scientific and Innovative Research, 2011"
ACQUIRED_TERRITORIES_ACT = "Acquired Territories (Merger) Act, 1960"
AYODHYA_ACT = "Acquisition of Certain Area at Ayodhya Act, 1993"
ACTUARIES_ACT = "Actuaries Act, 2006"
EXCISE_TEXTILES_ACT = "Additional Duties of Excise (Textiles and Textile Articles) Act, 1978"
ADMIN_TRIBUNALS_AMENDMENT_ACT = "Administrative Tribunals (Amendment) Act, 1986"
ADMIN_TRIBUNALS_ACT = "Administrative Tribunals Act, 1985"

GOLD_SET = [
    # --- Aadhaar Act, 2016 (7) ---
    {
        "query": "How can a resident obtain an Aadhaar number?",
        "gold_sections": [(AADHAAR_ACT, "3")],
    },
    {
        "query": "Can an Aadhaar number be reassigned to another person?",
        "gold_sections": [(AADHAAR_ACT, "4")],
    },
    {
        "query": (
            "What special measures must be taken to issue Aadhaar numbers "
            "to children, senior citizens and persons with disabilities?"
        ),
        "gold_sections": [(AADHAAR_ACT, "5")],
    },
    {
        "query": (
            "Can Aadhaar holders be required to update their demographic "
            "or biometric information?"
        ),
        "gold_sections": [(AADHAAR_ACT, "6")],
    },
    {
        "query": (
            "When can proof of Aadhaar be required for receiving a government "
            "subsidy, benefit or service?"
        ),
        "gold_sections": [(AADHAAR_ACT, "7")],
    },
    {
        "query": "Does having an Aadhaar number serve as proof of Indian citizenship or domicile?",
        "gold_sections": [(AADHAAR_ACT, "9")],
    },
    {
        "query": "What is the penalty for impersonating another person at the time of Aadhaar enrolment?",
        "gold_sections": [(AADHAAR_ACT, "34")],
    },

    # --- Abducted Persons (Recovery and Restoration) Continuance Act, 1955 (1) ---
    {
        "query": "When did the Abducted Persons (Recovery and Restoration) Continuance Act, 1955 come into force?",
        "gold_sections": [(ABDUCTED_PERSONS_ACT, "1")],
    },

    # --- Absorbed Areas (Laws) Act, 1954 (2) ---
    {
        "query": "How does the Absorbed Areas (Laws) Act, 1954 define an 'absorbed area'?",
        "gold_sections": [(ABSORBED_AREAS_ACT, "2")],
    },
    {
        "query": "Under the Absorbed Areas (Laws) Act, 1954, which laws get extended to the absorbed areas?",
        "gold_sections": [(ABSORBED_AREAS_ACT, "3")],
    },

    # --- Academy of Scientific and Innovative Research, 2011 (2) ---
    {
        "query": "Has the Academy of Scientific and Innovative Research been declared an institution of national importance?",
        "gold_sections": [(ACADEMY_ACT, "6")],
    },
    {
        "query": "What are the objects of the Academy of Scientific and Innovative Research?",
        "gold_sections": [(ACADEMY_ACT, "4")],
    },

    # --- Acquired Territories (Merger) Act, 1960 (1) ---
    {
        "query": "What happens to the acquired territories under the Acquired Territories (Merger) Act, 1960?",
        "gold_sections": [(ACQUIRED_TERRITORIES_ACT, "3")],
    },

    # --- Acquisition of Certain Area at Ayodhya Act, 1993 (2) ---
    {
        "query": "How does the Acquisition of Certain Area at Ayodhya Act, 1993 vest rights over the area in the Central Government?",
        "gold_sections": [(AYODHYA_ACT, "3")],
    },
    {
        "query": "What is the penalty for failing to hand over assets or documents relating to the acquired Ayodhya area?",
        "gold_sections": [(AYODHYA_ACT, "10")],
    },

    # --- Actuaries Act, 2006 (3) ---
    {
        "query": "Who is entitled to have their name entered in the register of Actuaries under the Actuaries Act, 2006?",
        "gold_sections": [(ACTUARIES_ACT, "6")],
    },
    {
        "query": "What disqualifies a person from having their name entered in the register of Actuaries?",
        "gold_sections": [(ACTUARIES_ACT, "11")],
    },

    # --- Additional Duties of Excise (Textiles and Textile Articles) Act, 1978 (1) ---
    {
        "query": "When did the Additional Duties of Excise (Textiles and Textile Articles) Act, 1978 come into force?",
        "gold_sections": [(EXCISE_TEXTILES_ACT, "1")],
    },

    # --- Administrative Tribunals (Amendment) Act, 1986 (1) ---
    {
        "query": "When did the Administrative Tribunals (Amendment) Act, 1986 come into force?",
        "gold_sections": [(ADMIN_TRIBUNALS_AMENDMENT_ACT, "1")],
    },

    # --- Administrative Tribunals Act, 1985 (1) ---
    {
        "query": "What jurisdiction, powers and authority does the Central Administrative Tribunal exercise under the Administrative Tribunals Act, 1985?",
        "gold_sections": [(ADMIN_TRIBUNALS_ACT, "14")],
    },
]

assert len(GOLD_SET) == 20, f"expected 20 gold examples, got {len(GOLD_SET)}"
print(f"Gold set size: {len(GOLD_SET)}")
print(f"Distinct Acts covered: {len({g['gold_sections'][0][0] for g in GOLD_SET})}")

Gold set size: 20
Distinct Acts covered: 10


In [ ]:
def recall_at_k(retrieved_sections: list[tuple], gold_sections: list[tuple]) -> float:
    gold_set = set(gold_sections)   # set for O(1) membership checks instead of scanning a list repeatedly
    # any(...) short-circuits True as soon as one retrieved section matches any gold section.
    hit = any(rs in gold_set for rs in retrieved_sections)
    return 1.0 if hit else 0.0

def mrr(retrieved_sections: list[tuple], gold_sections: list[tuple]) -> float:
    gold_set = set(gold_sections)
    # enumerate(..., start=1) -> rank 1, 2, 3... matching how humans count rank position.
    for rank, rs in enumerate(retrieved_sections, start=1):
        if rs in gold_set:
            return 1.0 / rank   # first (best) match found -- reciprocal of its rank
    return 0.0   # no match anywhere in the retrieved list

RETRIEVAL_TOP_N = 6

retrieval_scores = []
for g in GOLD_SET:
    try:
        reranked = rerank(g["query"], retrieve(g["query"]), top_n=RETRIEVAL_TOP_N)
        # Build (act_title, section) tuples from the reranked payloads, in rank order, to compare against gold.
        retrieved_sections = [(c["payload"]["act_title"], c["payload"]["section"]) for c in reranked]
        retrieval_scores.append({
            "query": g["query"],
            "gold_sections": g["gold_sections"],
            "top_hit": retrieved_sections[0] if retrieved_sections else None,
            f"recall@{RETRIEVAL_TOP_N}": recall_at_k(retrieved_sections, g["gold_sections"]),
            "mrr": mrr(retrieved_sections, g["gold_sections"]),
        })
    except Exception as e:
        # A single bad query (empty index, network hiccup, etc.) shouldn't kill the whole eval run --
        # record it as a zero-score failure and keep going so the rest of the gold set still gets scored.
        print(f"[retrieval eval] failed on query {g['query']!r}: {e}")
        retrieval_scores.append({
            "query": g["query"],
            "gold_sections": g["gold_sections"],
            "top_hit": None,
            f"recall@{RETRIEVAL_TOP_N}": 0.0,
            "mrr": 0.0,
        })

retrieval_df = pd.DataFrame(retrieval_scores)

print(f"Mean recall@{RETRIEVAL_TOP_N}: {retrieval_df[f'recall@{RETRIEVAL_TOP_N}'].mean():.3f}")
print(f"Mean MRR:       {retrieval_df['mrr'].mean():.3f}")
print(f"Misses (recall@{RETRIEVAL_TOP_N} == 0): {int((retrieval_df[f'recall@{RETRIEVAL_TOP_N}'] == 0).sum())} / {len(retrieval_df)}")

retrieval_df

retrieve() called for: 'How can a resident obtain an Aadhaar number?'
  embed query (dense): 0.82s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.01s
  mmr_select: 0.24s
retrieve() called for: 'Can an Aadhaar number be reassigned to another person?'
  embed query (dense): 0.40s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.01s
  mmr_select: 0.41s
retrieve() called for: 'What special measures must be taken to issue Aadhaar numbers to children, senior citizens and persons with disabilities?'
  embed query (dense): 0.45s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.01s
  mmr_select: 0.46s
retrieve() called for: 'Can Aadhaar holders be required to update their demographic or biometric information?'
  embed query (dense): 0.52s
  embed query (sparse): 0.00s
  qdrant dense search: 0.02s
  qdrant sparse search: 0.02s
  mmr_select: 0.64s
retrieve() called for: 'When can proof of

,query,gold_sections,top_hit,recall@6,mrr
0,How can a resident obtain an Aadhaar number?,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
1,Can an Aadhaar number be reassigned to another...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
2,What special measures must be taken to issue A...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
3,Can Aadhaar holders be required to update thei...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
4,When can proof of Aadhaar be required for rece...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
5,Does having an Aadhaar number serve as proof o...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
6,What is the penalty for impersonating another ...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0
7,When did the Abducted Persons (Recovery and Re...,[(Abducted Persons (Recovery and Restoration) ...,(Abducted Persons (Recovery and Restoration) C...,1.0,1.0
8,"How does the Absorbed Areas (Laws) Act, 1954 d...","[(Absorbed Areas (Laws) Act, 1954, 2)]","(Absorbed Areas (Laws) Act, 1954, 2)",1.0,1.0
9,"Under the Absorbed Areas (Laws) Act, 1954, whi...","[(Absorbed Areas (Laws) Act, 1954, 3)]","(Absorbed Areas (Laws) Act, 1954, 3)",1.0,1.0


top_hit — the #1 ranked result the pipeline actually returned after reranking

recall@6 — 1.0 if the correct section appeared anywhere in the top 6 reranked results, 0.0 if it didn't appear at all

mrr — Mean Reciprocal Rank: 1/rank of the correct section. 1.0 means it was ranked #1, 0.5 would mean it showed up at rank 2, 0.25 at rank 4, etc.



In [ ]:
from deepeval import evaluate
from deepeval.metrics import GEval, AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Reuse the same secret-resolution helper from Section 1 (works in Colab and outside it) instead of
# calling `userdata.get` directly -- that name only exists inside Colab and would NameError anywhere else.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY or get_secret("OPENAI_API_KEY") or ""
assert os.environ["OPENAI_API_KEY"], (
    "OPENAI_API_KEY is not set -- DeepEval's GEval/AnswerRelevancy/Faithfulness metrics use an "
    "OpenAI model as the judge by default, so this key is required even if the RAG pipeline itself "
    "is configured to use a different generation provider."
)

# GEval is DeepEval's "describe the rubric in plain English, let an LLM judge grade against it" metric.
citation_correctness = GEval(
    name="Citation Correctness",
    criteria=(
        "Check whether every legal claim in the actual output is backed by a citation "
        "(Act + Section) that is present in the retrieval context, and that the cited "
        "section actually supports the claim being made."
    ),
    # Tells DeepEval which fields of the test case the grading LLM is allowed to look at.
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.RETRIEVAL_CONTEXT,
    ],
)

# Built-in DeepEval metrics -- threshold=0.7 is the pass/fail cutoff each metric reports against.
answer_relevancy = AnswerRelevancyMetric(threshold=0.7)
faithfulness = FaithfulnessMetric(threshold=0.7)

test_cases = []
generation_failures = []
for g in GOLD_SET:
    try:
        result = app.invoke({
            "session_id": "eval-session", "raw_query": g["query"],
            "act_filter": None, "provider": "openai",
        })
        test_cases.append(LLMTestCase(
            input=g["query"],
            actual_output=result["answer"],
            retrieval_context=[result["context"]],   # DeepEval expects a list of context strings
        ))
    except Exception as e:
        # Same principle as the retrieval loop above: one failed pipeline run shouldn't stop the
        # whole eval -- log it and keep building test cases for the rest of the gold set.
        print(f"[generation eval] app.invoke failed on query {g['query']!r}: {e}")
        generation_failures.append({"query": g["query"], "error": str(e)})

print(f"Built {len(test_cases)}/{len(GOLD_SET)} test cases "
      f"({len(generation_failures)} pipeline failures)")

eval_results = evaluate(test_cases, [citation_correctness, answer_relevancy, faithfulness])
eval_results

/tmp/ipykernel_854/3627425808.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


sub_queries: ['How can a resident obtain an Aadhaar number?']
retrieve() called for: 'How can a resident obtain an Aadhaar number?'
  embed query (dense): 0.39s
  embed query (sparse): 0.00s
  qdrant dense search: 0.01s
  qdrant sparse search: 0.01s
  mmr_select: 0.61s
sub_queries: ['Can an Aadhaar number be reassigned to another person?', 'If an Aadhaar number can be reassigned, what is the process for doing so?']
retrieve() called for: 'Can an Aadhaar number be reassigned to another person?'
  embed query (dense): 0.47s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.01s
  mmr_select: 0.33s
retrieve() called for: 'If an Aadhaar number can be reassigned, what is the process for doing so?'
  embed query (dense): 0.41s
  embed query (sparse): 0.00s
  qdrant dense search: 0.00s
  qdrant sparse search: 0.00s
  mmr_select: 0.59s
sub_queries: ['What special measures are required to issue Aadhaar numbers to children?', 'What special measures are required 

✨ You're running DeepEval's latest Citation Correctness [GEval] Metric! (using gpt-5.4, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.4, strict=False, async_mode=True)...

...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            What special measures must be taken to issue Aadhaar numbers to children, senior       │
│  │                       citizens and persons with disabilities?                                                │
│  │     Actual Output:    Special measures for the issuance of Aadhaar numbers to certain categories of          │
│  │                       persons, including women, children, senior citizens, and persons with disabilities,    │
│  │                       are mandated by Section 5 of the Aadhaar (Targeted Delivery of Financial and other     │
│  │                       Subsidies, Benefits and Services) Act, 2016. The Authority is required to take         │
│  │                       special measures to issue Aadhaar numbers to these individuals, as well as to          │
│  │                       unskilled and unorganised workers, nomadic tribes, and any other specified             │
│  │                       categories of individuals who may not have a permanent dwelling house or may be        │
│  │                       specified by regulations.                                                              │
│  │                                                                                                              │
│  │                       (Reference: Aadhaar (Targeted Delivery of Financial and other Subsidies, Benefits      │
│  │                       and Services) Act, 2016 | Section: 5)                                                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                       ┃ Score ┃ Threshold ┃ Reason                                        │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Citation Correctness [GEval] │ 1.00  │ 0.50      │ The response addresses the exact issue in     │
│              │                              │       │           │ the i...                                      │
│        FAIL  │ Answer Relevancy             │ 0.62  │ 0.70      │ The score is 0.62 because the answer          │
│              │                              │       │           │ appears to address the question about         │
│              │                              │       │           │ special measures for issuing Aadhaar to       │
│              │                              │       │           │ children, senior citizens, and persons with   │
│              │                              │       │     

⚠ WARNING: No hyperparameters logged.
» ]8;id=9998;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 23.53s | token cost: 1.0852700000000004 USD)
» Test Results (20 total tests):
   » Pass Rate: 90.0% | Passed: 18 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_9', success=True, metrics_data=[MetricData(name='Citation Correctness [GEval]', threshold=0.5, success=True, score=1.0, reason='The response makes one relevant legal claim: that Section 3 extends the Acts listed in column 1 of each Schedule, along with rules, notifications, orders, schemes, forms, and bye-laws made thereunder, as in force in the absorbing State named in the Schedule heading, to the absorbed areas. It provides a complete citation to the Absorbed Areas (Laws) Act, 1954, Section 3. That citation appears in the retrieval context and the quoted substance matches Section 3 closely without adding unsupported details or citing the wrong provision.', strict_mode=False, evaluation_model='gpt-5.4', error=None, evaluation_cost=0.007952500000000001, input_tokens=1453, output_tokens=288, verbose_logs='Criteria:\nCheck whether every legal claim in the actual output is backed by a citation (Act + Section) that is present in the

In [ ]:
# --- Combined summary: retrieval layer + generation layer side by side ---
#
# `eval_results.test_results` (DeepEval >=1.x) holds one entry per test case, each with a `.metrics_data`
# list of per-metric scores -- flatten that into a tidy per-query, per-metric table and merge it with the
# retrieval scores computed above so pass/fail on both layers is visible for the same 20 gold queries.

generation_rows = []
for tr in getattr(eval_results, "test_results", []):
    row = {"query": tr.input}
    for md in tr.metrics_data:
        row[f"{md.name} score"] = round(md.score, 3) if md.score is not None else None
        row[f"{md.name} passed"] = md.success
    generation_rows.append(row)

generation_df = pd.DataFrame(generation_rows)

summary_df = retrieval_df.merge(generation_df, on="query", how="left")

print("=== Retrieval layer ===")
print(f"Mean recall@{RETRIEVAL_TOP_N}: {retrieval_df[f'recall@{RETRIEVAL_TOP_N}'].mean():.3f}")
print(f"Mean MRR:       {retrieval_df['mrr'].mean():.3f}")

print("\n=== Generation layer ===")
for col in generation_df.columns:
    if col.endswith(" passed"):
        pass_rate = generation_df[col].mean()
        print(f"{col.replace(' passed', '')} pass rate: {pass_rate:.0%} ({generation_df[col].sum()}/{len(generation_df)})")

# Persist the full per-query breakdown for offline review / regression tracking across notebook runs.
summary_df.to_csv("eval_results_summary.csv", index=False)
print("\nSaved per-query breakdown to eval_results_summary.csv")

summary_df

=== Retrieval layer ===
Mean recall@6: 1.000
Mean MRR:       0.960

=== Generation layer ===
Citation Correctness [GEval] pass rate: 100% (20/20)
Answer Relevancy pass rate: 90% (18/20)
Faithfulness pass rate: 100% (20/20)

Saved per-query breakdown to eval_results_summary.csv


,query,gold_sections,top_hit,recall@6,mrr,Citation Correctness [GEval] score,Citation Correctness [GEval] passed,Answer Relevancy score,Answer Relevancy passed,Faithfulness score,Faithfulness passed
0,How can a resident obtain an Aadhaar number?,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,1.000,True,1.000,True,1.0,True
1,Can an Aadhaar number be reassigned to another...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,0.827,True,1.000,True,1.0,True
2,What special measures must be taken to issue A...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,1.000,True,0.625,False,1.0,True
3,Can Aadhaar holders be required to update thei...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,0.985,True,1.000,True,1.0,True
4,When can proof of Aadhaar be required for rece...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,0.918,True,1.000,True,1.0,True
5,Does having an Aadhaar number serve as proof o...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,1.000,True,1.000,True,1.0,True
6,What is the penalty for impersonating another ...,[(Aadhaar (Targeted Delivery of Financial and ...,(Aadhaar (Targeted Delivery of Financial and o...,1.0,1.0,1.000,True,1.000,True,1.0,True
7,When did the Abducted Persons (Recovery and Re...,[(Abducted Persons (Recovery and Restoration) ...,(Abducted Persons (Recovery and Restoration) C...,1.0,1.0,1.000,True,1.000,True,1.0,True
8,"How does the Absorbed Areas (Laws) Act, 1954 d...","[(Absorbed Areas (Laws) Act, 1954, 2)]","(Absorbed Areas (Laws) Act, 1954, 2)",1.0,1.0,1.000,True,1.000,True,1.0,True
9,"Under the Absorbed Areas (Laws) Act, 1954, whi...","[(Absorbed Areas (Laws) Act, 1954, 3)]","(Absorbed Areas (Laws) Act, 1954, 3)",1.0,1.0,1.000,True,1.000,True,1.0,True
